# Capstone — mirrors your deployed research paper

## 1. Question

**Core Research Question:**
*Given thousands of published URLs across heterogeneous client domains, can machine-learned ranking over observable search and engagement signals outperform fixed heuristic rules in prioritizing decaying content for editorial refresh review?*

**Decision & Operational Context:**
- **Target Decision:** Which 20 to 50 specific pages should a content editorial team audit and refresh first each sprint cycle?
- **Actor:** Content strategists and digital editors.
- **Action Taken:** Targeted content refresh, metadata/snippet CTR optimization, heading expansion, or internal re-linking.
- **Cost of Wrong Decision:** Recommending healthy/low-demand pages wastes constrained human capacity; failing to surface severely decaying high-exposure pages causes lasting, compounding organic traffic loss.

In [1]:
RESEARCH_QUESTION = "Can learned models beat static rules at prioritizing decaying search content?"
TARGET_METRIC = "Precision@50 on Client-Holdout Validation"
PRIMARY_ACTOR = "Content Operations & Editorial Team"

print(f"Research Question: {RESEARCH_QUESTION}")
print(f"Target Benchmark Metric: {TARGET_METRIC}")
print(f"Decision Consumer: {PRIMARY_ACTOR}")

Research Question: Can learned models beat static rules at prioritizing decaying search content?
Target Benchmark Metric: Precision@50 on Client-Holdout Validation
Decision Consumer: Content Operations & Editorial Team


## 2. Data

**Data Sources & Provenance:**
- **Dataset:** Anonymized starter snapshot (`data/raw/content_refresh_anonymized.csv`), encompassing 30,000 unique URLs across 32 clients over a trailing 90-day window.
- **Observable Telemetry Only:** Features are composed strictly of observable Google Search Console (GSC) and Google Analytics 4 (GA4) metrics, plus public content metadata.
- **Excluded Fields & Public Safety:**
  - `trend_direction` & `trend_pct`: Excluded due to fatal circular label leakage.
  - `*_last_30d` metrics: Excluded to prevent evaluation lookahead leakage.
  - `content_id` & `client_id`: Hashed pseudonym identifiers; reserved strictly for joining and client-holdout validation splits.
  - No client names, URLs, page titles, or raw search queries exist in the dataset.

In [2]:
import os
import pandas as pd
import numpy as np

data_path = "../../data/raw/content_refresh_anonymized.csv" if os.path.exists("../../data/raw/content_refresh_anonymized.csv") else "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print(f"Verified Scope: {len(df):,} total pages across {df['client_id'].nunique()} distinct clients.")
print(f"Median Content Age: {df['content_age_days'].median():.0f} days.")
print(f"Rate scaling audit: Median CTR = {df['ctr'].median():.2f}% (confirmed x100 scale).")

Verified Scope: 30,000 total pages across 32 distinct clients.
Median Content Age: 236 days.
Rate scaling audit: Median CTR = 0.07% (confirmed x100 scale).


## 3. Methodology

**System Architecture & Assumptions:**
1. **Target Formulation:** $\text{is\_declining\_label} = (\text{trend\_direction} == \text{'down'})$ ($54.2\%$ portfolio base rate).
2. **Transparent Heuristic Baseline:**
   $$\text{Score}_{\text{baseline}} = 100 \times (0.40 \cdot S_{\text{visibility}} + 0.30 \cdot S_{\text{freshness}} + 0.25 \cdot S_{\text{position}})$$
3. **Machine Learning Model:** Random Forest Classifier (100 trees, max depth 10) trained over log-transformed search volumes, engagement rates, and content metadata.
4. **Validation Design:** Strict **Client-Holdout Group Split** (20% held-out clients, 0 domain overlap).

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

# Filter to mature content with search exposure
clean_df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy().reset_index(drop=True)
clean_df["is_declining_label"] = (clean_df["trend_direction"].str.lower() == "down").astype(int)

# Transform heavy-tailed signals
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    clean_df[f"log_{col}"] = np.log1p(clean_df[col].clip(lower=0))

feature_cols = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
]
cat_cols = ["competition_level", "content_type", "main_intent", "age_tier", "position_tier"]

X = pd.concat([clean_df[feature_cols].fillna(0), pd.get_dummies(clean_df[cat_cols].fillna("unknown"), drop_first=True)], axis=1)
y = clean_df["is_declining_label"].values

# Client-holdout split
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=clean_df["client_id"]))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Training on {len(X_train):,} pages ({clean_df.iloc[train_idx]['client_id'].nunique()} clients)")
print(f"Testing on  {len(X_test):,} pages ({clean_df.iloc[test_idx]['client_id'].nunique()} unseen clients)")

Training on 23,837 pages (25 clients)
Testing on  6,163 pages (7 unseen clients)


## 4. Results (vs baseline)

We evaluate predictions on one held-out group of unseen clients across Precision@20, Precision@50, ROC-AUC, and the holdout positive rate. This is a single group split, not cross-validation or out-of-fold evaluation.

In [4]:
from sklearn.metrics import roc_auc_score

def p_at_k(scores, y_true, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(y_true)[order[:k]].mean()

# Baseline on test set
vis_t = np.log1p(clean_df.iloc[test_idx]["impressions_90d"].clip(lower=0)) / np.log1p(30000)
fresh_t = (clean_df.iloc[test_idx]["days_since_last_update"] / 180.0).clip(0, 1.0)
test_position = clean_df.iloc[test_idx]["avg_position"]
pos_t = np.where((test_position > 3) & (test_position <= 20), 1.0,
        np.where((test_position > 0) & (test_position <= 3), 0.5, 0.1))
base_scores = (0.40 * vis_t + 0.30 * fresh_t + 0.25 * pos_t) * 100.0

# Fit Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]

# Summary benchmark
results_table = pd.DataFrame([
    {"Method": "Heuristic Rule Baseline", "Precision@20": p_at_k(base_scores, y_test, 20), "Precision@50": p_at_k(base_scores, y_test, 50), "ROC-AUC": roc_auc_score(y_test, base_scores)},
    {"Method": "Random Forest (Learned)", "Precision@20": p_at_k(rf_probs, y_test, 20), "Precision@50": p_at_k(rf_probs, y_test, 50), "ROC-AUC": roc_auc_score(y_test, rf_probs)}
])

print("--- HEADLINE CAPSTONE RESULTS (Client-Holdout Evaluation) ---")
display(results_table.round(3))

p50_base = results_table.loc[0, "Precision@50"]
p50_rf = results_table.loc[1, "Precision@50"]
print(f"Holdout positive rate (base rate): {y_test.mean():.3f}")
print(f"Measured on this single client-holdout split, the learned model achieves {p50_rf:.3f} Precision@50 vs {p50_base:.3f} for the heuristic ({p50_rf/p50_base:.1f}x ratio). This is directional ranking evidence, not a causal effect.")

--- HEADLINE CAPSTONE RESULTS (Client-Holdout Evaluation) ---


,Method,Precision@20,Precision@50,ROC-AUC
0,Heuristic Rule Baseline,0.20,0.34,0.529
1,Random Forest (Learned),0.45,0.58,0.609


Holdout positive rate (base rate): 0.511
Measured on this single client-holdout split, the learned model achieves 0.580 Precision@50 vs 0.340 for the heuristic (1.7x ratio). This is directional ranking evidence, not a causal effect.


## 5. Limitations

**Explicit Scientific Boundaries:**
1. **No Causal Recovery Guarantee:**
    On this one client-holdout split, the learned ranking placed more declining pages in its top 50 than the corrected heuristic. This is directional evidence on one sample, not proof that the model will generalize to every client or that a refresh will recover traffic. Causal uplift requires prospective randomized experimentation.
2. **Observational SERP Nuances:**
   Search traffic drops can occur when Google introduces zero-click answer cards or AI Overviews for high-volume keywords, an external factor not captured in numeric page-level telemetry.
3. **Tabular Scope:**
   The model learns from numeric traffic, ranking, and engagement telemetry; it does not process raw semantic content or factual text depth.

In [5]:
print("LIMITATIONS SUMMARY:")
print("1. Observational decision-support only (no causal claims without intervention trials).")
print("2. Google SERP feature shifts (AI overviews, zero-click answers) are unobserved.")
print("3. Retraining required when Google core updates trigger portfolio distribution shifts.")

LIMITATIONS SUMMARY:
1. Observational decision-support only (no causal claims without intervention trials).
2. Google SERP feature shifts (AI overviews, zero-click answers) are unobserved.
3. Retraining required when Google core updates trigger portfolio distribution shifts.


## 6. Ranked recommendations

We blend model probability ($70\%$) with baseline exposure utility ($30\%$) to create an exploratory full-portfolio review queue. Queue scores for training clients are in-sample; the ranked list is not a validation result.

In [6]:
# Compute full blended scores
vis_full = np.log1p(clean_df["impressions_90d"].clip(lower=0)) / np.log1p(30000)
fresh_full = (clean_df["days_since_last_update"] / 180.0).clip(0, 1.0)
position_full = clean_df["avg_position"]
pos_full = np.where((position_full > 3) & (position_full <= 20), 1.0,
           np.where((position_full > 0) & (position_full <= 3), 0.5, 0.1))
base_full = (0.40 * vis_full + 0.30 * fresh_full + 0.25 * pos_full) * 100.0

clean_df["model_prob"] = rf.predict_proba(X)[:, 1]
clean_df["final_refresh_score"] = 100.0 * (0.70 * clean_df["model_prob"] + 0.30 * (base_full / 100.0))

def get_action(r):
    if r["model_prob"] < 0.40 and r["final_refresh_score"] < 45: return "monitor"
    if 0 < r["word_count"] < 1200 and r["impressions_90d"] >= 250: return "expand_and_refresh"
    if r["impressions_90d"] >= 500 and 0 < r["avg_position"] <= 20 and r["ctr"] < 0.5: return "refresh_and_review_ctr"
    if r["sessions_90d"] >= 30 and (r["engagement_rate"] < 30 or r["scroll_rate"] < 30): return "refresh_and_review_engagement"
    return "refresh"

clean_df["recommended_action"] = clean_df.apply(get_action, axis=1)
capstone_queue = clean_df.sort_values("final_refresh_score", ascending=False).reset_index(drop=True)
capstone_queue["rank"] = capstone_queue.index + 1

action_summary = capstone_queue["recommended_action"].value_counts().rename_axis("Recommended action").reset_index(name="Pages")
print(f"Action queue contains {len(capstone_queue):,} eligible pages. Aggregate action counts:")
display(action_summary)

Action queue contains 30,000 eligible pages. Aggregate action counts:


,Recommended action,Pages
0,refresh,10877
1,refresh_and_review_ctr,9428
2,monitor,6428
3,refresh_and_review_engagement,3204
4,expand_and_refresh,63


## 7. Artifacts the paper embeds

We generate aggregate charts in `work/outputs/` for the deployed paper. Row-level queue records and pseudonymous IDs are not published.

In [7]:
import matplotlib.pyplot as plt
from pathlib import Path

repo_root = Path(data_path).resolve().parents[2]
output_dir = repo_root / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# 1. Feature Importance Plot
fig, ax = plt.subplots(figsize=(8, 4))
top_feats = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(8)
top_feats.plot(kind="barh", color="#2ca02c", ax=ax)
ax.set_title("Top 8 Predictors of Search Traffic Decay")
ax.set_xlabel("Relative Importance")
plt.tight_layout()
feat_chart_path = output_dir / "capstone_feature_importance.png"
fig.savefig(feat_chart_path, dpi=150)
plt.close(fig)
print(f"Generated and saved: {feat_chart_path}")

# 2. Precision Comparison Chart
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Heuristic Rule", "Random Forest"], [p50_base, p50_rf], color=["#7f7f7f", "#1f77b4"], width=0.5)
ax.set_ylabel("Precision@50")
ax.set_ylim(0, 1.0)
ax.set_title("Decision Precision Lift: Rule vs Machine Learning")
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.03, f"{yval:.3f}", ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
prec_chart_path = output_dir / "capstone_precision_comparison.png"
fig.savefig(prec_chart_path, dpi=150)
plt.close(fig)
print(f"Generated and saved: {prec_chart_path}")

Generated and saved: C:\Users\divya\Desktop\FL_Machine_learning\work\outputs\capstone_feature_importance.png
Generated and saved: C:\Users\divya\Desktop\FL_Machine_learning\work\outputs\capstone_precision_comparison.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.